In [7]:
import pandas as pd
from statsmodels.formula.api import ols
from scipy import stats

In [2]:
df = pd.read_csv('study.csv')
df.head()

,study_hours,material_type,score
0,71,강의,95
1,34,독학,63
2,91,도서,95
3,80,독학,80
4,40,강의,79


### 1.통계요약

In [5]:
model = ols('score~study_hours+material_type',data=df).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  score   R-squared:                       0.969
Model:                            OLS   Adj. R-squared:                  0.968
Method:                 Least Squares   F-statistic:                     991.9
Date:                Thu, 12 Nov 2026   Prob (F-statistic):           4.42e-72
Time:                        16:04:28   Log-Likelihood:                -238.89
No. Observations:                 100   AIC:                             485.8
Df Residuals:                      96   BIC:                             496.2
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
=======================================================================================
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept              59.2111      0.799     74.147      0.000      57.626      60.796
material_type[T.도서]    -8.6696      0.678    -12.778      0.000     -10.016      -7.323
material_type[T.독학]   -17.6129      0.634    -27.790      0.000     -18.871     -16.355
study_hours             0.4839      0.011     43.810      0.000       0.462       0.506
==============================================================================
Omnibus:                        1.754   Durbin-Watson:                   2.173
Prob(Omnibus):                  0.416   Jarque-Bera (JB):                1.216
Skew:                           0.231   Prob(JB):                        0.544
Kurtosis:                       3.280   Cond. No.                         228.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### 2.학습시간과 점수의 상관계수

In [6]:
r2 = df['study_hours'].corr(df['score'])
print("상관계수 :", r2)

상관계수 : 0.8468695225074608


### 3.학습시간과 점수의  상관계수 및 p-value

In [12]:
result = stats.pearsonr(df['study_hours'], df['score'])
print("상관계수 :", result.statistic)
print("p-value :", "{:.10f}".format(result.pvalue))

상관계수 : 0.8468695225074604
p-value : 0.0000000000


### 4.회귀모델의 결정계수 및 수정된 결정계수

In [15]:
print("결정계수 :", model.rsquared)
print("수정된 결정계수 :", model.rsquared_adj)

결정계수 : 0.9687460807310152
수정된 결정계수 : 0.9677693957538595


### 5.회귀 모델에서 회귀계수(기울기,절편)을 구하시오.

In [19]:
print("절편 :", model.params['Intercept'])
print("기울기(학습시간) :", model.params['study_hours'])

절편 : 59.21113684065031
기울기(학습시간) : 0.483902956545778


### 6.학습시간의 회귀계수가 통계적으로 유의한지 검정(p-value)

In [22]:
print("p-value :", "{:.10f}".format(model.pvalues['study_hours']))

p-value : 0.0000000000


### 7.학습시간 65시간, 공부방식 '강의'인 데이터의 예상 점수

In [26]:
new_data = pd.DataFrame({
    'study_hours' : [65],
    'material_type' : ['강의'],
})

pred = model.predict(new_data)
print("예상점수 :", pred[0])

예상점수 : 90.66482901612588


### 8.회귀모델의 잔차의 제곱합을 구하시오.

In [27]:
df['잔차'] = df['score'] - model.predict(df)
sse = sum(df['잔차']**2)
print("잔차의 제곱합 : ", sse)

잔차의 제곱합 :  695.8572611130093


### 9.회귀모델의 MSE를 구하시오.

In [28]:
mse = (df['잔차']**2).mean()
print("MSE :", mse)

MSE : 6.958572611130092


### 10.각 변수별 95% 신뢰구간

In [29]:
conf = model.conf_int(alpha=0.05)
conf

,0,1
Intercept,57.626008,60.796266
material_type[T.도서],-10.016331,-7.322792
material_type[T.독학],-18.870962,-16.354825
study_hours,0.461978,0.505828


### 11.학습시간 65시간, 공부방식 '강의'의 95% 신뢰구간과 예측구간을 구하시오.

In [30]:
pred = model.get_prediction(new_data)
summary = pred.summary_frame(alpha=0.05)

print('예측값 :', summary.loc[0, 'mean'])
print(f'신뢰구간 : {summary.loc[0,'mean_ci_lower']} ~ {summary.loc[0,'mean_ci_upper']}')
print(f'예측구간 : {summary.loc[0,'obs_ci_lower']} ~ {summary.loc[0,'obs_ci_upper']}')

예측값 : 90.66482901612588
신뢰구간 : 89.79235652775351 ~ 91.53730150449825
예측구간 : 85.24989324017854 ~ 96.07976479207322
